## `semisparse` Tutorial 01: The semi-sparse OTI module.

```
Contributors:
    Mauricio Aristizabal

Date of creation:  Sep 28 2026
```

### Introduction

`pyoti.semisparse` is a third way to store Order Truncated Imaginary (OTI) numbers, next to the
sparse types of `pyoti.sparse` and the static dense types. It keeps the same idea (a real part plus
imaginary coefficients along directions $\epsilon_{i_1}\epsilon_{i_2}\cdots$) and the same Python API,
so scripts written for `pyoti.sparse` run with

```python
import pyoti.semisparse as oti
```

The difference is the storage, and therefore which problems are fast. This notebook walks through the
scalars, the matrices, the Gauss-point types and the finite element helpers of the module, and
compares them with `pyoti.sparse` wherever that is useful.

In [1]:
import itertools
import time
from math import comb

import numpy as np

import pyoti.sparse as sparse    # the reference implementation
import pyoti.semisparse as oti   # this tutorial

oti.set_printoptions(terms_print=-1)   # print every term (the setting is shared with pyoti.sparse)
np.set_printoptions(precision=6, suppress=True)

### What is semi-sparse?

A semi-sparse number is **dense over a sorted set of active bases**. If a number depends on the bases
$\{2, 5\}$ and has truncation order 3, it stores *every* direction of order 1 to 3 over those two bases
as one contiguous vector of coefficients, zeros included. There are no index lists: which direction
each entry holds is fixed by the numbering.

The numbering is the *local colexicographic* order over the active bases: the directions of order $p$
over $k$ bases are $N_p(k)=\binom{k+p-1}{p}$, and the directions that only use the first $k'$ bases
are exactly the first $N_p(k')$ entries. Multiplication is then a table-driven loop over contiguous
memory, and matrices can share **one** active set (a *structure of arrays*, `oarrss`) so that every
direction is a plain real matrix that can be handed to BLAS.

In [2]:
x = 1.5 + oti.e(2, order=3) + 4 * oti.e([2, 5])

print(x)
print("active bases :", x.active_bases)
print("order        :", x.order)
print("nonzeros     :", x.nnz, " density of the stored slots:", round(x.density, 3))

k = len(x.active_bases)
print("stored coefficients per order:", [comb(k + p - 1, p) for p in range(1, x.order + 1)])

1.5 + 1 * e([2]) + 4 * e([2,5])
active bases : (2, 5)
order        : 3
nonzeros     : 3  density of the stored slots: 0.222
stored coefficients per order: [2, 3, 4]


Directions are written exactly as in `pyoti.sparse`: `e(i)`, `e([i, j])`, `e([[i, m], j])` for
$\epsilon_i^m\epsilon_j$. One difference: in the semi-sparse module a Python list or tuple is
*always* a list of bases. The raw pair `(index, order)` of the global numbering (what `pyoti.core.imdir`
returns) must be wrapped in `oti.rawdir`, and every function that takes a direction accepts it.

In [3]:
d = oti.rawdir(4, 2)          # global index 4 of order 2
print(d, "is the direction of bases", d.bases)
print(oti.rawdir.from_direction([2, 5]), "<- the same idea, built from bases")

print(x.get_im([2, 5]), x.get_im(oti.rawdir.from_direction([2, 5])))
print(oti.imdir([[2, 2], 5]), "<- [global index, order] of e([[2, 2], 5]), as in pyoti.sparse")

rawdir(4, 2) is the direction of bases (2, 3)
rawdir(11, 2) <- the same idea, built from bases
4.0 4.0
[22, 3] <- [global index, order] of e([[2, 2], 5]), as in pyoti.sparse


#### When does it pay off?

Semi-sparse spends memory on zeros to gain regular, branch-free kernels. It wins when the numbers
are *dense over their active bases* (few bases, high order, or many bases at low order), which is the
typical case of a sensitivity analysis with a handful of parameters. It loses when a number
depends on many bases but only through a few directions: then almost every stored slot is zero.
A small timing of one product in each situation (the numbers depend on your machine):

In [4]:
def build(module, order, directions):
    number = module.number(1.0, order=order)
    for j, direction in enumerate(directions):
        number.set_im(1.0 + 0.01 * j, direction)
    return number


def timing(number, repetitions):
    number * number                                   # warm-up: builds the multiplication tables once
    start = time.perf_counter()
    for _ in range(repetitions):
        product = number * number
    return (time.perf_counter() - start) / repetitions * 1e6   # microseconds


# Dense over 5 bases, order 4: every direction is present.
dense_dirs = [list(d) for p in range(1, 5)
              for d in itertools.combinations_with_replacement(range(1, 6), p)]
# Few directions: 60 bases, order 4, but only the 60 first-order terms and 10 pairs.
few_dirs = [[b] for b in range(1, 61)] + [[b, b + 1] for b in range(1, 11)]

print(f"{'case':36s} {'sparse [us]':>12s} {'semisparse [us]':>16s}   semisparse storage")
for name, dirs, order, reps in (("dense: 5 bases, order 4", dense_dirs, 4, 300),
                                ("few directions: 60 bases, order 4", few_dirs, 4, 10)):
    s_num = build(sparse, order, dirs)
    m_num = build(oti, order, dirs)
    print(f"{name:36s} {timing(s_num, reps):12.1f} {timing(m_num, reps):16.1f}   "
          f"{m_num.memory_bytes / 1e3:.1f} kB for {m_num.nnz} nonzeros")

case                                  sparse [us]  semisparse [us]   semisparse storage
dense: 5 bases, order 4                       4.3              0.9   1.0 kB for 126 nonzeros
few directions: 60 bases, order 4            91.5            455.6   5083.2 kB for 71 nonzeros


There is no automatic switch between the two representations: choose the one that fits the problem
(conversions in both directions are shown below). Results agree to round-off, so the sparse module is
also the natural way to check a semi-sparse computation.

### Scalars

The scalar type is `ssotinum`. It is created with the same functions as in `pyoti.sparse`:
`e`, `number`, `zero`, `one`, and ordinary arithmetic with floats and other OTI numbers.

In [5]:
E = 200e9 + oti.e(1, order=3)
nu = 0.3 + oti.e(2, order=3)

lam = E * nu / ((1 + nu) * (1 - 2 * nu))        # first Lame parameter
print(type(lam).__name__, lam.active_bases, lam.order)
print("value            :", lam.real)
print("d lam / dE       :", lam.get_deriv(1))
print("d lam / dnu      :", lam.get_deriv(2))
print("d2 lam / dE dnu  :", lam.get_deriv([1, 2]))
print("d3 lam / dnu^3   :", lam.get_deriv([[2, 3]]))

ssotinum (1, 2) 3
value            : 115384615384.61537
d lam / dE       : 0.5769230769230769
d lam / dnu      : 872781065088.7574
d2 lam / dE dnu  : 4.3639053254437865
d3 lam / dnu^3   : 125140051118658.31


Compare against `pyoti.sparse`: the same expression with the sparse `e` gives the same derivatives.
Any semi-sparse number can be converted with `to_sparse()`, and `ssotinum(...)` accepts a sparse
scalar.

In [6]:
E_s = 200e9 + sparse.e(1, order=3)
nu_s = 0.3 + sparse.e(2, order=3)
lam_s = E_s * nu_s / ((1 + nu_s) * (1 - 2 * nu_s))

for direction in ([1], [2], [1, 2], [2, 2], [[2, 3]]):
    print(f"{str(direction):10s} semisparse {lam.get_deriv(direction): .12e}"
          f"   sparse {lam_s.get_deriv(direction): .12e}")

back = oti.ssotinum(lam_s)              # sparse -> semi-sparse
print("round trip identical:", np.isclose(back.get_deriv([1, 2]), lam.get_deriv([1, 2]), rtol=1e-13))
print(type(lam.to_sparse()).__name__)   # semi-sparse -> sparse

[1]        semisparse  5.769230769231e-01   sparse  5.769230769231e-01
[2]        semisparse  8.727810650888e+11   sparse  8.727810650888e+11
[1, 2]     semisparse  4.363905325444e+00   sparse  4.363905325444e+00
[2, 2]     semisparse  8.272644515248e+12   sparse  8.272644515248e+12
[[2, 3]]   semisparse  1.251400511187e+14   sparse  1.251400511187e+14
round trip identical: True
sotinum


The elementary functions (`sin`, `cos`, `tan`, `asin`, `acos`, `atan`, `sinh`, `cosh`, `tanh`, `asinh`,
`acosh`, `atanh`, `exp`, `log`, `log10`, `logb`, `sqrt`, `cbrt`, `erf`) and the power `x ** p` work on
scalars and arrays; a plain float argument returns a float.

In [7]:
t = 0.4 + oti.e(1, order=4) + 0.5 * oti.e([1, 2])
f = oti.exp(oti.sin(t)) / oti.sqrt(1 + t * t)
g = oti.pow(t, 1.5) + t ** 0.5 + 2 ** t          # float and OTI exponents

print(f)
print("d4 f / dx1^4 :", f.get_deriv([[1, 4]]))
print(oti.sin(0.5), "<- a float in, a float out")

1.37054 + 0.789754 * e([1]) - 0.467102 * e([[1,2]]) + 0.394877 * e([1,2]) - 0.234476 * e([[1,3]]) - 0.467102 * e([[1,2],2]) + 0.172013 * e([[1,4]]) - 0.351713 * e([[1,3],2]) - 0.116776 * e([[1,2],[2,2]])
d4 f / dx1^4 : 4.128313067704638
0.479425538604203 <- a float in, a float out


#### Reading and changing coefficients

`get_im(dir)` returns the *coefficient* of a direction and `get_deriv(dir)` the *derivative* (the
coefficient times the product of the factorials of the exponents). `set_im` and `set_deriv` write them,
growing the active set if the direction uses a new base. `x[[index, order]]` and `x[rawdir(...)]` read
and write by raw direction, as the sparse scalar does.

In [8]:
y = 3.0 + oti.e([1, 1], order=3)              # 3 + e1^2
print("coefficient of e1^2 :", y.get_im([[1, 2]]))
print("derivative  d2y/dx1^2:", y.get_deriv([[1, 2]]))       # 2! times the coefficient

y.set_deriv(10.0, [[1, 2]])                                   # derivative 10 -> coefficient 5
print("after set_deriv     :", y.get_im([[1, 2]]), y.get_deriv([[1, 2]]))

y.set_im(2.0, [7])                                            # a new base joins the active set
print("active bases       :", y.active_bases, " (raw read:", y[oti.rawdir(6, 1)], ")")

coefficient of e1^2 : 1.0
derivative  d2y/dx1^2: 2.0
after set_deriv     : 5.0 10.0
active bases       : (1, 7)  (raw read: 2.0 )


#### Extracting parts of a number

- `extract_im(dir)` / `extract_deriv(dir)`: divide by a direction (collect the multiples of `dir`).
- `truncate(dir)`: zero a direction and every direction containing it.
- `truncate_order(p)`: drop the orders $\geq p$. `get_order_im(p)`: keep only order $p$.

In [9]:
z = 1.0 + oti.e(1, order=3) + 2 * oti.e([[1, 2]]) + 3 * oti.e([[1, 2], 2]) + 4 * oti.e(3)   # 1 + e1 + 2 e1^2 + 3 e1^2 e2 + 4 e3

print("extract_im([1])    :", z.extract_im([1]))
print("extract_deriv([1]) :", z.extract_deriv([1]))
print("truncate([2])      :", z.truncate([2]))          # removes e1^2 e2 (it contains e2)
print("truncate_order(2)  :", z.truncate_order(2))
print("get_order_im(2)    :", z.get_order_im(2))

extract_im([1])    : 1 + 2 * e([1]) + 3 * e([1,2])
extract_deriv([1]) : 1 + 4 * e([1]) + 6 * e([1,2])
truncate([2])      : 1 + 1 * e([1]) + 4 * e([3]) + 2 * e([[1,2]])
truncate_order(2)  : 1 + 1 * e([1]) + 4 * e([3])
get_order_im(2)    :  + 2 * e([[1,2]])


#### The active set

Bases are **never dropped automatically**. If a coefficient cancels, its base stays in the active set with
zero coefficients (this keeps every operation branch-free); `compact()` removes the bases that are
not needed.

In [10]:
w = (1.0 + oti.e(1, order=2) + oti.e(2, order=2)) - oti.e(2, order=2)

print(w, w.active_bases, w.get_active_bases())
print("compact() ->", w.compact().active_bases)
print("nnz:", w.nnz, " actual_order:", w.actual_order, " get_nnz_full():", w.get_nnz_full())
print(w.short_repr())
print(w.long_repr())

1 + 1 * e([1]) (1, 2) [1, 2]
compact() -> (1,)
nnz: 2  actual_order: 1  get_nnz_full(): [2, 6, array([1, 0]), array([2, 3])]
ssotinum(1.0, nnz: 2, order: 1)
ssotinum(1.0, nnz: 2, alloc: 6, actual order: 1, truncation order: 2, bases: (1, 2)
  - Order 1->   nnz: 1  size: 2 
  - Order 2->   nnz: 0  size: 3 
)


### Arrays

There are two matrix layouts. Both are created by the module functions and behave alike:

| type | layout | active set | best for |
|---|---|---|---|
| `oarrss` (default) | structure of arrays, column-major: one real matrix per direction | **one** set shared by all elements | linear algebra (`dot`, `solve`, `inv`, `det`), FEM |
| `arrss` | array of structures, row-major: one `ssotinum` per element | one set **per element** | matrices whose elements depend on different bases |

`oti.zeros`, `oti.ones`, `oti.eye` and `oti.array` return `oarrss`. `to_aos()` / `to_soa()` convert between the layouts (going to SoA takes the union of the element sets), and `from_sparse` / `to_sparse` convert from and to `pyoti.sparse`.

In [11]:
A = oti.array([[1 + oti.e(1, order=2), 2.0],
               [3.0,                   4 + oti.e(2, order=2)]])

print(type(A).__name__, A.shape, "order", A.order, "bases", A.active_bases)
print("real part:\n", A.real)
print("d/dx1:\n", A.get_deriv(1))
print("d/dx2:\n", A.get_deriv(2))

B = oti.arrss.from_sparse(A.to_sparse())    # AoS: every element keeps its own active set
print(type(B).__name__, [B[i, j].active_bases for i in range(2) for j in range(2)])
print("to SoA (union of the sets):", A.to_aos().to_soa().active_bases, " to sparse:", type(B.to_sparse()).__name__)

oarrss (2, 2) order 2 bases (1, 2)
real part:
 [[1. 2.]
 [3. 4.]]
d/dx1:
 [[1. 0.]
 [0. 0.]]
d/dx2:
 [[0. 0.]
 [0. 1.]]
arrss [(1,), (), (), (2,)]
to SoA (union of the sets): (1, 2)  to sparse: matso


Creators keep the signatures of `pyoti.sparse`: `zeros(shape, nbases, order, nip)` (an integer shape is
a column vector), `ones`, `eye`, `array` (nested lists of floats and OTI scalars, or NumPy arrays,
where a 1-D input is a column), `zero`, `one` and `number`. The extra `bases=` argument of `zeros`
fixes the active set. `nbases` is only a capacity hint of the sparse types and has no effect here.

In [12]:
print(oti.zeros(3).shape, oti.zeros((2, 4), order=2).order, oti.zeros((2, 2), bases=[1, 3]).active_bases)
print(oti.eye(3).real)
print(oti.ones((2, 3)).real)
print(oti.array(np.arange(1.0, 4.0)).shape, "<- 1-D input is a column")

(3, 1) 2 (1, 3)
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[[1. 1. 1.]
 [1. 1. 1.]]
(3, 1) <- 1-D input is a column


#### Indexing and assignment

Indexing follows `pyoti.sparse.matso`: two integers give an element (a scalar copy), anything else
gives a block (a new array); `A[i]` is the row block `A[i, :]`. Assignment takes floats, OTI
scalars, OTI arrays of the block's shape, and NumPy arrays.

**The truncation order rises to the order of the value assigned.** Finite element assembly writes
`f[i, 0] = f[i, 0] + term` into an array created with order 0, and no derivative of `term` may be lost.

In [13]:
f = oti.zeros((4, 1))
print("order of f before:", f.order)

term = 5.0 + oti.e(1, order=3) + oti.e([2, 3], order=3)
f[1, 0] = f[1, 0] + term
f[3, 0] = 2.0

print("order of f after :", f.order, " bases:", f.active_bases)
print(f[1, 0])
print("the untouched entries were zero-extended:", f[0, 0].order, f[0, 0].real)

M = oti.zeros((3, 4), order=2)
M[0, 0:4:2] = oti.array([[1.0 + oti.e(1, order=2), 2.0]])      # a strided slice from an array
M[1:3, 1] = 7.5                                                  # a float over a block
M[2, 3] = oti.e(2, order=2)
print(M.real)
print(M[0, 0:2].shape, M[:, 1].shape, M[-1, -1].get_deriv(2), "<- negative indices count from the end")

order of f before: 0
order of f after : 3  bases: (1, 2, 3)
5 + 1 * e([1]) + 1 * e([2,3])
the untouched entries were zero-extended: 3 0.0
[[1.  0.  2.  0. ]
 [0.  7.5 0.  0. ]
 [0.  7.5 0.  0. ]]
(1, 2) (3, 1) 1.0 <- negative indices count from the end


#### Matrix products and linear algebra

`dot` (or `@`), `transpose` (or `.T`), `inv`, `det` and `solve`. The real part is factorized once with
LAPACK (LU) and every derivative order is obtained by block back-substitution, so the real matrix must be
non-singular.

In [14]:
K = oti.array([[4.0 + oti.e(1, order=3), 1.0,   0.0],
               [1.0,   3.0 + 0.5 * oti.e(1, order=3), 1.0],
               [0.0,   1.0,   2.0 + oti.e(2, order=3)]])
b = oti.array([[1.0], [2.0], [3.0 + oti.e(2, order=3)]])

u = oti.solve(K, b)
print("u        :", u.real.ravel())
print("du/dx1   :", u.get_deriv(1).ravel())
print("du/dx2   :", u.get_deriv(2).ravel())

residual = oti.dot(K, u) - b
print("max |K u - b| over every derivative up to order 3:",
      max(abs(residual.get_deriv(d)).max() for d in ([1], [2], [1, 1], [1, 2], [[2, 2]], [[1, 3]])))

print("det(K)   :", oti.det(K).real, " d det / dx1:", oti.det(K).get_deriv(1))
print("inv(K) @ K = I:", np.allclose((oti.inv(K) @ K).real, np.eye(3)))

u        : [0.222222 0.111111 1.444444]
du/dx1   : [-0.055556  0.       -0.      ]
du/dx2   : [-0.024691  0.098765 -0.271605]
max |K u - b| over every derivative up to order 3: 1.0408340855860843e-17
det(K)   : 18.0  d det / dx1: 9.0
inv(K) @ K = I: True


The block solvers of the sparse module are built from two truncated products, and both exist here:
`trunc_dot(i, A, j, B)` is the order-$(i+j)$ part of $A_i B_j$, and `dot_product(a, b)` is the sum of
elementwise products of two arrays of the same size (a scalar).

In [15]:
print("order-1 part of K u  :", oti.trunc_dot(1, K, 0, u).get_deriv(1).ravel())
print("dot_product(u, u)    :", oti.dot_product(u, u).real, " (real check:", float(u.real.ravel() @ u.real.ravel()), ")")

order-1 part of K u  : [0.222222 0.055556 0.      ]
dot_product(u, u)    : 2.148148148148149  (real check: 2.148148148148149 )


#### Direction blocks as NumPy views

An `oarrss` keeps each direction as a real matrix, **column-major**, in one buffer. `get_block(dir)`
returns a writable Fortran-ordered NumPy view of a block (`0` is the real part), so the coefficients can be
inspected or filled without copies. The view keeps the array alive, but it is invalidated by anything that
grows the array (a new base, a higher order). `get_block` raises `KeyError` for a direction outside the
active set.

In [16]:
S = oti.zeros((2, 3), bases=[1, 2], order=2)
real = S.get_block(0)
d1 = S.get_block([1])
print(real.shape, "column-major:", real.flags["F_CONTIGUOUS"], " owns its data:", real.flags["OWNDATA"])

real[:] = np.arange(6.0).reshape(2, 3)     # fill the real part directly
d1[0, 2] = 5.0                              # coefficient of e1 in element (0, 2)
print(S[0, 2])
print(S.real)

try:
    S.get_block([3])
except KeyError as error:
    print("KeyError:", error)

(2, 3) column-major: True  owns its data: False
2
[[0. 1. 2.]
 [3. 4. 5.]]
KeyError: "direction is not in the array's active set"


### Saving and reading

`save` writes a scalar or an array (either layout) to a compact binary file with a version field, the
orders, the active bases (zero-coefficient bases included) and the coefficients; `read` returns the same
object.

In [17]:
import os
import tempfile

with tempfile.TemporaryDirectory() as folder:
    path = os.path.join(folder, "K.ssoti")
    oti.save(K, path)
    K2 = oti.read(path)
    print(type(K2).__name__, K2.shape, K2.active_bases, K2.order, "size on disk:", os.path.getsize(path), "bytes")
    print("identical:", np.array_equal(K2.get_deriv([1, 2]), K.get_deriv([1, 2])))

    oti.save(lam, os.path.join(folder, "lam.ssoti"))
    print(oti.read(os.path.join(folder, "lam.ssoti")).get_deriv([1, 2]) == lam.get_deriv([1, 2]))

oarrss (3, 3) (1, 2) 3 size on disk: 792 bytes
identical: True
True


### Gauss-point types

Numerical integration needs the same quantity at every integration point. The semi-sparse module has
two batched types for that: `ssotife` (a scalar at `nip` points) and `oarrssfe` (a matrix at `nip` points).
All points share **one** active set, and every direction is one $nip\times n_{rows}\times n_{cols}$ real
block, so an operation on all points is a single batched kernel. They are created with the usual creators
and the `nip` argument, and mix freely with plain scalars and arrays.

In [18]:
w = oti.number(0.5, nip=3)                         # weights of a 3-point rule (all equal here)
print(w)

fe = oti.zeros((2, 2), nip=3, order=2)             # a 2x2 matrix at 3 points
fe = fe + oti.array([[1.0 + oti.e(1, order=2), 2.0], [0.0, 3.0]])
fe[0, 1] = fe[0, 1] * oti.e(2, order=2)            # scalar at each point: 2 * e2

print(type(fe).__name__, "nip =", fe.nip, "shape =", fe.shape, "bases =", fe.get_active_bases())
print(fe.get_ip(1))                                # the matrix at point 1 (an oarrss)
print("integral over the points:\n", oti.gauss_integrate(fe, w))

ssotife(nip=3, bases=[], order=0, real=[0.5, 0.5, 0.5])
oarrssfe nip = 3 shape = (2, 2) bases = [1, 2]
matso< shape: (2, 2), 
 - Column 0
(0,0) 1 + 1 * e([1])
(1,0) 0
 - Column 1
(0,1)  + 2 * e([2])
(1,1) 3
>
integral over the points:
 matso< shape: (2, 2), 
 - Column 0
(0,0) 1.5 + 1.5 * e([1])
(1,0) 0
 - Column 1
(0,1)  + 3 * e([2])
(1,1) 4.5
>


The module functions work on Gauss-point operands and act on every point: `dot`, `inv`, `det`,
`transpose`, `sum`, `sub`, `mul`, `div`, and the elementary functions.

In [19]:
G = oti.dot(fe, oti.transpose(fe))
print(type(G).__name__, G.nip, G.get_ip(0).real.ravel(), " det at point 2:", oti.det(fe)[2].real)

oarrssfe 3 [1. 0. 0. 9.]  det at point 2: 3.0


### Finite elements with `pyoti.fem`

`pyoti.fem` picks the OTI algebra it computes with. `set_global_algebra(pyoti.semisparse)` makes the mesh
coordinates, the element helpers (`N`, `Nx`, `Ny`, `dV`, ...) and everything derived from them
semi-sparse. Below, the stiffness matrix of one unit-square quad4 element with the Young modulus
$E = 2\cdot10^{11} + \epsilon_1$ and the Poisson ratio $\nu = 0.3+\epsilon_2$ is assembled the way
`examples/python/fem_twc.py` assembles the thick-walled cylinder, with a routine that never mentions the
algebra module by name.

In [20]:
import pyoti.fem as fem


def quad4_stiffness(alg, order=2):
    fem.set_global_algebra(alg)
    E = 200e9 + alg.e(1, order=order)
    nu = 0.3 + alg.e(2, order=order)
    D = (E / ((1 + nu) * (1 - 2 * nu))) * alg.array([
        [1 - nu, nu, 0],
        [nu, 1 - nu, 0],
        [0, 0, (1 - 2 * nu) / 2],
    ])

    x = alg.array(np.array([0.0, 1.0, 1.0, 0.0]))       # unit square, counter-clockwise
    y = alg.array(np.array([0.0, 0.0, 1.0, 1.0]))
    z = alg.zeros((4, 1))

    element = fem.element[3]                              # gmsh type 3: 4-node quadrilateral
    element.end()
    element.allocate(intorder=1, nbases=2, order=order)   # 2x2 Gauss points
    element.allocate_spatial(2, compute_Jinv=True)
    element.set_coordinates(x, y, z, np.arange(4))
    element.compute_jacobian()

    nb = element.nbasis
    Ke = alg.zeros((2 * nb, 2 * nb))
    for ip in range(element.nip):
        Nx, Ny, dV = element.Nx.get_ip(ip), element.Ny.get_ip(ip), element.dV[ip]
        B = alg.zeros((3, 2 * nb))
        B[0, 0:2 * nb:2] = Nx
        B[1, 1:2 * nb:2] = Ny
        B[2, 0:2 * nb:2] = Ny
        B[2, 1:2 * nb:2] = Nx
        Ke += alg.dot(B.T, alg.dot(D, B)) * dV
    fem.end_elements()
    return Ke, E


Ke, E = quad4_stiffness(oti)
print(type(Ke).__name__, Ke.shape, "bases", Ke.active_bases)

eigenvalues = np.linalg.eigvalsh(Ke.real)
print("rigid-body modes (eigenvalues ~ 0):", int(np.sum(eigenvalues < 1e-6 * eigenvalues.max())), "of", len(eigenvalues))
print("dKe/dE == Ke/E:", np.allclose(Ke.get_deriv(1), Ke.real / E.real, rtol=1e-12))

oarrss (8, 8) bases (1, 2)
rigid-body modes (eigenvalues ~ 0): 3 of 8
dKe/dE == Ke/E: True


In [21]:
Ke_sparse, _ = quad4_stiffness(sparse)
fem.set_global_algebra(oti)

for direction in ([1], [2], [1, 2], [[2, 2]]):
    difference = np.abs(Ke.get_deriv(direction) - Ke_sparse.get_deriv(direction)).max()
    scale = np.abs(Ke_sparse.get_deriv(direction)).max()
    print(f"{str(direction):9s} max relative difference to pyoti.sparse: {difference / scale:.2e}")

[1]       max relative difference to pyoti.sparse: 4.81e-17
[2]       max relative difference to pyoti.sparse: 2.34e-16
[1, 2]    max relative difference to pyoti.sparse: 3.40e-16
[[2, 2]]  max relative difference to pyoti.sparse: 3.43e-16


### OTI sparse matrices: assembling and solving large systems

Finite element matrices are large and mostly zero, and every nonzero is an OTI number. Two containers
with the API of `pyoti.sparse` hold them:

- `lil_matrix((n, m))`: for *assembly*. `K[i, j]` reads and writes an entry (a scalar copy), so the
  usual `K[i, j] = K[i, j] + value` accumulates element contributions in any order.
- `csr_matrix`: for *solving*. `K.tocsr()` fixes the sparsity pattern. All entries share **one** active
  set, and each direction is one real vector over the nonzeros, so the real part is a SciPy CSR matrix
  (`K.real`) that the SciPy factorizations use directly.

The example is a chain of $n=300$ springs with stiffness $E = 10+\epsilon_1$ fixed at one end and loaded
with $F = 1+\epsilon_2$ at the other, so the solution carries derivatives with respect to $E$ and $F$;
the tip displacement is $nF/E$.

In [22]:
def chain(alg, n=300, order=3):
    E = 10.0 + alg.e(1, order=order)
    F = 1.0 + alg.e(2, order=order)

    K = alg.lil_matrix((n, n))
    for i in range(n):                                   # element i couples nodes i and i+1
        K[i, i] = K[i, i] + 2.0 * E
        if i + 1 < n:
            K[i, i + 1] = K[i, i + 1] - E
            K[i + 1, i] = K[i + 1, i] - E
    K[n - 1, n - 1] = K[n - 1, n - 1] - E                # free end: one spring less

    f = alg.zeros((n, 1))
    f[n - 1, 0] = f[n - 1, 0] + F
    return K, f


K, f = chain(oti)
print(type(K).__name__, K.shape, " entry K[0, 0] =", K[0, 0], " entry K[0, 5] =", K[0, 5])

Kc = K.tocsr()
print(type(Kc).__name__, Kc.shape, "nonzeros:", Kc.nnz, " order:", Kc.order, " active bases:", Kc.get_active_bases())

lil_matrix (300, 300)  entry K[0, 0] = 20 + 2 * e([1])  entry K[0, 5] = 0
csr_matrix (300, 300) nonzeros: 898  order: 3  active bases: [1]


`csr_matrix.real` is a `scipy.sparse.csr_matrix` **view** of the real coefficients (no copy), so any
SciPy routine can inspect it, and `K.get_deriv(d)`, `K.get_im(d)` return the same for a derivative
direction. `K @ x` and `dot` multiply a matrix by an array, order by order.

In [23]:
import scipy.sparse.linalg as spla

R = Kc.real
print(type(R).__module__, type(R).__name__, R.shape)
print("same buffer on every call:", R.data.ctypes.data == Kc.real.data.ctypes.data)
print("largest eigenvalues of the real part:", spla.eigsh(R, k=2, return_eigenvectors=False))
print("d K / d E has the pattern of K:", Kc.get_deriv(1).nnz == Kc.nnz)

x = oti.ones((300, 1)) + oti.e(2, order=3)
y = Kc @ x                                    # OTI sparse matrix times OTI array
print(type(y).__name__, y.shape, y[0, 0], "|", (K.tocsr().dot(x))[1, 0])

scipy.sparse._csr csr_matrix (300, 300)
same buffer on every call: True
largest eigenvalues of the real part: [39.995628 39.998907]
d K / d E has the pattern of K: True
oarrss (300, 1) 10 + 1 * e([1]) + 10 * e([2]) + 1 * e([1,2]) | 0


#### Solving

`solve(K, f, solver='SuperLU', solver_args={})` factorizes the **real part once** with SciPy/SuperLU
(other choices: `'spilu'`, `'cholesky'` and `'umfpack'` when the packages are installed) and solves for
every derivative order with block triangular solves, using the truncated products `trunc_dot` and `trunc_sub`.

In [24]:
u = oti.solve(Kc, f, solver='SuperLU')
print(type(u).__name__, u.shape, "order", u.order, "bases", u.get_active_bases())
print("tip displacement u[-1]      :", u.real[-1, 0], " (exact n F / E = 30)")
print("d u_tip / dF                :", u.get_deriv(2)[-1, 0])
print("d u_tip / dE                :", u.get_deriv(1)[-1, 0], " (exact -n F / E^2 = -3)")
print("d2 u_tip / dE dF            :", u.get_deriv([1, 2])[-1, 0], " (exact -n / E^2 = -3)")

residual = Kc @ u - f
print("largest residual over derivatives up to 3:",
      max(np.abs(residual.get_deriv(d)).max() for d in ([], [1], [2], [1, 2], [[1, 2]], [[1, 3]], [[2, 2]])))

try:
    v = oti.solve(Kc, f, solver='cholesky')      # K is symmetric positive definite
    print("cholesky agrees with SuperLU:", np.allclose(v.get_deriv([1, 2]), u.get_deriv([1, 2]), rtol=1e-12))
except ImportError as error:
    print("solver='cholesky' needs scikit-sparse:", error)

oarrss (300, 1) order 3 bases [1, 2]
tip displacement u[-1]      : 30.000000000001805  (exact n F / E = 30)
d u_tip / dF                : 30.000000000001805
d u_tip / dE                : -3.000000000000329  (exact -n F / E^2 = -3)
d2 u_tip / dE dF            : -3.000000000000329  (exact -n / E^2 = -3)
largest residual over derivatives up to 3: 1.2789769243681803e-13
cholesky agrees with SuperLU: True


#### Comparison with `pyoti.sparse`

The same assembly and solve with the sparse algebra: identical results, and a look at the times of the
two stages (Python-level assembly is dominated by the per-call overhead of both; the solve is where the
kernels differ).

In [25]:
def run(alg, n=300, order=3):
    t0 = time.perf_counter(); K, f = chain(alg, n, order)
    t1 = time.perf_counter(); Kc = K.tocsr()
    t2 = time.perf_counter(); u = alg.solve(Kc, f, solver='SuperLU')
    t3 = time.perf_counter()
    return u, (t1 - t0, t2 - t1, t3 - t2)


u_s, t_s = run(sparse)
u_m, t_m = run(oti)

print(f"{'stage':10s} {'sparse [ms]':>12s} {'semisparse [ms]':>16s}")
for name, a, b in zip(("assembly", "tocsr", "solve"), t_s, t_m):
    print(f"{name:10s} {a * 1e3:12.1f} {b * 1e3:16.1f}")

for d in ([], [1], [2], [1, 2], [[1, 2]], [[1, 3]], [2, 2]):
    a = u_s.real if not d else u_s.get_deriv(d)
    b = u_m.real if not d else u_m.get_deriv(d)
    print(f"{str(d):10s} max relative difference: {np.abs(a - b).max() / max(np.abs(a).max(), 1e-300):.2e}")

stage       sparse [ms]  semisparse [ms]
assembly            0.4              0.3
tocsr               0.1              0.1
solve               1.7              0.2
[]         max relative difference: 0.00e+00
[1]        max relative difference: 0.00e+00
[2]        max relative difference: 0.00e+00
[1, 2]     max relative difference: 0.00e+00
[[1, 2]]   max relative difference: 0.00e+00
[[1, 3]]   max relative difference: 0.00e+00
[2, 2]     max relative difference: 0.00e+00


### The same script runs with either algebra

The thick-walled cylinder of `examples/python/fem_twc.py` is written against the shared API only:

```python
import pyoti.sparse, pyoti.semisparse
from fem_twc import run

result_sparse = run(pyoti.sparse,     [8, 8], order=2)
result_semi   = run(pyoti.semisparse, [8, 8], order=2)
```

`run(algebra, ndivs, order, perturb_geometry=False)` builds the gmsh mesh, assembles $K$ and $f$,
imposes the symmetry conditions, solves, and compares every derivative with the Lamé solution.
Swapping the module is all it takes, which makes the pair an easy way to compare the two
representations on a real problem.

### Summary

| | `pyoti.sparse` | `pyoti.semisparse` |
|---|---|---|
| storage | index + value of each nonzero direction | dense over a shared, sorted active set |
| good for | many bases, few directions | few bases or dense expansions, matrix algebra |
| scalar | `sotinum` | `ssotinum` |
| matrices | `matso` | `oarrss` (SoA, default), `arrss` (AoS) |
| Gauss points | `sotife`, `matsofe` (nip objects) | `ssotife`, `oarrssfe` (batched blocks) |
| directions | list/tuple of bases | list/tuple of bases, `rawdir(index, order)` for raw pairs |